In [6]:
bacth_id=''

StatementMeta(, 80202277-0337-4006-b2e8-3ba958c7f615, 8, Finished, Available, Finished, False)

In [7]:
# ================================================================
# 🔥 MANEJO DE ERRORES - BRONZE
# ================================================================
import traceback
from datetime import datetime
from zoneinfo import ZoneInfo

from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp, date_format

CATALOG = "Prueba Tecnica Dataknow"
BRONZE_LH = "LH_BRONZE_PRUEBA"
SCHEMA = "dbo"

PIPELINE_NAME = "BRONZE"

BOGOTA_TZ = ZoneInfo("America/Bogota")

pipeline_errors = []

def ejecutar_con_manejo_errores(nombre_tarea, funcion):
    try:
        funcion()

        pipeline_errors.append(Row(
            pipeline=PIPELINE_NAME,
            tarea=nombre_tarea,
            estado="OK",
            mensaje="Ejecutado correctamente",
            detalle_error="",
            timestamp=datetime.now(BOGOTA_TZ).strftime("%Y-%m-%d %H:%M:%S")
        ))

        print(f"✅ {nombre_tarea}")

    except Exception as e:
        pipeline_errors.append(Row(
            pipeline=PIPELINE_NAME,
            tarea=nombre_tarea,
            estado="ERROR",
            mensaje=str(e)[:500],
            detalle_error=traceback.format_exc()[:2000],
            timestamp=datetime.now(BOGOTA_TZ).strftime("%Y-%m-%d %H:%M:%S")
        ))

        print(f"⚠️ Error en '{nombre_tarea}': {str(e)[:200]}")
        print("→ Registrado. Continuando...")

def guardar_errores_pipeline():
    if pipeline_errors:
        df = spark.createDataFrame(pipeline_errors)

        df = df.withColumn(
            "_fec_registro",
            date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss")
        )

        df.write.format("delta") \
          .mode("append") \
          .option("mergeSchema", "true") \
          .saveAsTable(f"`{CATALOG}`.{BRONZE_LH}.{SCHEMA}.pipeline_error_log")

        ok = len([e for e in pipeline_errors if e.estado == "OK"])
        err = len([e for e in pipeline_errors if e.estado == "ERROR"])

        print(f"\n📋 pipeline_error_log: {ok} OK, {err} errores")

StatementMeta(, 80202277-0337-4006-b2e8-3ba958c7f615, 9, Finished, Available, Finished, False)

## Código final

In [8]:
# ===========================
# Bronze Ingestion - Microsoft Fabric
# ===========================

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, current_timestamp, lit, year, month, dayofmonth, to_utc_timestamp, from_utc_timestamp, date_format
)
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, DoubleType
)

from delta.tables import DeltaTable
from datetime import datetime
from datetime import timedelta

import logging
import time

# ---------------------------
# Configuración de logs
# ---------------------------
# LOGS en zona America/Bogota (pegar justo después de los imports)
import logging
from datetime import datetime, timedelta, timezone

# Intentar usar zoneinfo (Py3.9+). Si no está, intentar pytz. Si tampoco, fallback UTC-5.
try:
    from zoneinfo import ZoneInfo  # Python 3.9+
    BOGOTA_TZ = ZoneInfo("America/Bogota")
except Exception:
    try:
        import pytz
        BOGOTA_TZ = pytz.timezone("America/Bogota")
    except Exception:
        # fallback simple: zona fija UTC-5 (Colombia no usa DST)
        BOGOTA_TZ = timezone(timedelta(hours=-5))

class TZFormatter(logging.Formatter):
    def __init__(self, fmt=None, datefmt=None, tz=None):
        super().__init__(fmt=fmt, datefmt=datefmt)
        self.tz = tz

    def formatTime(self, record, datefmt=None):
        # record.created es epoch time
        if hasattr(self.tz, "localize"):
            # pytz timezone
            ct = datetime.fromtimestamp(record.created, tz=timezone.utc).astimezone(self.tz)
        else:
            # zoneinfo or datetime.timezone
            ct = datetime.fromtimestamp(record.created, tz=self.tz if self.tz else None)
        if datefmt:
            return ct.strftime(datefmt)
        return ct.isoformat()

# Crear logger y handler (reemplaza configuración previa)
logger = logging.getLogger("BronzeIngest")
logger.setLevel(logging.INFO)

if logger.hasHandlers():
    logger.handlers.clear()

ch = logging.StreamHandler()
fmt = "%(asctime)s - %(levelname)s - %(message)s"
datefmt = "%Y-%m-%d %H:%M:%S"
ch.setFormatter(TZFormatter(fmt=fmt, datefmt=datefmt, tz=BOGOTA_TZ))
logger.addHandler(ch)
logger.propagate = False

spark = SparkSession.builder.appName("FinBank_Bronze_Ingestion").getOrCreate()
# Forzar zona horaria de sesión Spark a Colombia (recomendado)
spark.conf.set("spark.sql.session.timeZone", "America/Bogota")

spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# ---------------------------
# RUTAS
# ---------------------------
LAKEHOUSE_ID = "ade8ea43-56d4-4c3c-a5ba-110969bbd4c1"
WORKSPACE_ID = "41ce1f56-ff90-4d43-8757-58ea03e3849d"

LAKEHOUSE_AUDIT = f"abfss://{LAKEHOUSE_ID}@onelake.dfs.fabric.microsoft.com/{WORKSPACE_ID}/Files/log_ingesta"

# ---------------------------
# CATÁLOGO ORIGEN
# ---------------------------
CATALOG = "Prueba Tecnica Dataknow"
DATABASE = "LH_BRONZE_PRUEBA"
SCHEMA_ORIGEN = "dbo"

# ---------------------------
# DESTINO BRONZE
# ---------------------------
BRONZE_SCHEMA = "dbo"
BRONZE_PREFIX = "BRZ_"

# ---------------------------
# Configuración de tablas
# ---------------------------
TABLES_CONFIG = {
    "tb_clientes_core": {
        "primary_key": "id_cli",
        "incremental": True,
        "incremental_col": "fec_alta",
        "bronze_name": "TB_CLIENTES_CORE"
    },
    "tb_productos_cat": {
        "primary_key": None,
        "incremental": False,
        "incremental_col": None,
        "bronze_name": "TB_PRODUCTOS_CAT"
    },
    "tb_mov_financieros": {
        "primary_key": None,
        "incremental": False,
        "incremental_col": None,
        "bronze_name": "TB_MOV_FINANCIEROS"
    },
    "tb_obligaciones": {
        "primary_key": "id_oblig",
        "incremental": True,
        "incremental_col": "fec_desembolso",
        "bronze_name": "TB_OBLIGACIONES"
    },
    "tb_sucursales_red": {
        "primary_key": None,
        "incremental": False,
        "incremental_col": None,
        "bronze_name": "TB_SUCURSALES_RED"
    },
    "tb_comisiones_log": {
        "primary_key": "id_comision",
        "incremental": True,
        "incremental_col": "fec_cobro",
        "bronze_name": "TB_COMISIONES_LOG"
    }
}

# ---------------------------
# Schema del log de auditoría
# ---------------------------
AUDIT_SCHEMA = StructType([
    StructField("execution_time", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("records_processed", LongType(), True),
    StructField("size_bytes", LongType(), True),
    StructField("duration_sec", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True),
    StructField("load_type", StringType(), True),
    StructField("watermark_value", StringType(), True)
])


# ---------------------------
# Función auxiliar
# ---------------------------
def get_delta_table_size(table_name):
    """Calcula el tamaño de una tabla gestionada"""
    try:
        detail_df = spark.sql(f"DESCRIBE DETAIL {table_name}")
        size_bytes = detail_df.select("sizeInBytes").collect()[0][0]
        return size_bytes if size_bytes else 0
    except Exception as e:
        logger.warning(f"  ⚠️  No se pudo calcular tamaño: {e}")
        return 0


# ---------------------------
# Clase de ingesta Bronze
# ---------------------------
class BronzeIngestionFabric:

    def __init__(self, batch_id=None):
        self.batch_id = batch_id or f"batch_{int(datetime.now().timestamp())}"
        self.execution_start = datetime.now()
        self._ensure_audit_directory()
        logger.info(f"🚀 Iniciando ingesta Bronze | Batch: {self.batch_id}")

    def _ensure_audit_directory(self):
        """Crea la tabla de auditoría si no existe"""
        logger.info("\n" + "=" * 80)
        logger.info("📁 Verificando estructura de auditoría...")
        logger.info("=" * 80)
        try:
            spark.read.format("delta").load(LAKEHOUSE_AUDIT).limit(0).collect()
            logger.info(f"✅ Tabla de auditoría existe")
        except Exception:
            logger.info(f"⚠️  Creando tabla de auditoría...")
            try:
                empty_df = spark.createDataFrame([], AUDIT_SCHEMA)
                empty_df.write.format("delta").mode("overwrite").save(LAKEHOUSE_AUDIT)
                logger.info(f"✅ Tabla de auditoría creada")
            except Exception as e:
                logger.error(f"❌ Error: {e}")
                raise
        logger.info("=" * 80 + "\n")

    def _get_bronze_table_name(self, bronze_name):
        return f"{BRONZE_SCHEMA}.{BRONZE_PREFIX}{bronze_name}"

    def _get_last_watermark(self, bronze_table_name, incremental_col):
        """Obtiene el último watermark desde la tabla Bronze destino"""
        try:
            if spark.catalog.tableExists(bronze_table_name):
                result = spark.sql(
                    f"SELECT MAX({incremental_col}) as max_val FROM {bronze_table_name}"
                ).collect()[0][0]
                if result is not None:
                    logger.info(f"  📌 Último watermark: {result}")
                    return result
        except Exception as e:
            logger.warning(f"  ⚠️  No se pudo obtener watermark: {e}")
        logger.info(f"  📌 Sin watermark previo → carga completa inicial")
        return None

    def read_from_catalog(self, table_name):
        """Lee desde el catálogo ORIGEN"""
        full_table_name = f"`{CATALOG}`.`{DATABASE}`.`{SCHEMA_ORIGEN}`.`{table_name}`"
        logger.info(f"  📖 Leyendo ORIGEN: {full_table_name}")
        return spark.sql(f"SELECT * FROM {full_table_name}")

    def read_incremental_from_catalog(self, table_name, incremental_col, watermark):
        """Lee solo registros nuevos/modificados"""
        full_table_name = f"`{CATALOG}`.`{DATABASE}`.`{SCHEMA_ORIGEN}`.`{table_name}`"
        if watermark is not None:
            logger.info(f"  📖 INCREMENTAL: {incremental_col} > '{watermark}'")
            df = spark.sql(f"""
                SELECT * FROM {full_table_name}
                WHERE {incremental_col} > '{watermark}'
            """)
        else:
            logger.info(f"  📖 Primera carga COMPLETA")
            df = spark.sql(f"SELECT * FROM {full_table_name}")
        return df
    
    def add_audit_columns(self, df, source_system='SQL_FINBANK'):
        """
        Agrega columnas de metadatos:
          - 3 de auditoría: _ingest_timestamp, _source_system, _batch_id
          - 3 de partición: _ingest_year, _ingest_month, _ingest_day
            (derivadas de _ingest_timestamp para particionamiento)
        """
        # current_timestamp() -> devuelve la hora según spark.sql.session.timeZone (ya la tienes en America/Bogota)
        ingest_ts_local_ts = current_timestamp()  # timestamp en zona de sesión (Bogotá)
        # convertir esa hora local a UTC para guardar el timestamp oficial
        ingest_ts_utc = to_utc_timestamp(ingest_ts_local_ts, "America/Bogota")
        # representación legible en Bogotá (string)
        ingest_ts_local_str = date_format(ingest_ts_local_ts, "yyyy-MM-dd HH:mm:ss")


        df = (
            df.withColumn("_ingest_timestamp", ingest_ts_local_str)           # UTC (tipo timestamp)
            .withColumn("_source_system", lit(source_system))
            .withColumn("_batch_id", lit(self.batch_id))
            .withColumn("_ingest_year", year(ingest_ts_local_ts))
            .withColumn("_ingest_month", month(ingest_ts_local_ts))
            .withColumn("_ingest_day", dayofmonth(ingest_ts_local_ts))
        )
        return df
        

    def ingest_table(self, table_name, table_config):
        """Ingesta de una tabla hacia Tables/ con prefijo BRZ_"""
        bronze_name = table_config["bronze_name"]
        bronze_table = self._get_bronze_table_name(bronze_name)
        primary_key = table_config["primary_key"]
        is_incremental = table_config["incremental"]
        incremental_col = table_config.get("incremental_col")
        t0 = time.time()
        load_type = "FULL"
        watermark_value = None

        logger.info(f"\n{'=' * 70}")
        logger.info(f"📥 ORIGEN:  {SCHEMA_ORIGEN}.{table_name}")
        logger.info(f"📤 DESTINO: {bronze_table}")
        logger.info(f"📋 MODO:    {'INCREMENTAL' if is_incremental else 'FULL OVERWRITE'}")
        logger.info(f"{'=' * 70}")

        try:
            if is_incremental and incremental_col:
                watermark = self._get_last_watermark(bronze_table, incremental_col)
                watermark_value = str(watermark) if watermark else None
                df = self.read_incremental_from_catalog(table_name, incremental_col, watermark)
                load_type = "INCREMENTAL" if watermark else "INITIAL_FULL"
            else:
                df = self.read_from_catalog(table_name)
                load_type = "FULL_OVERWRITE"

            record_count = df.count()
            logger.info(f"  📊 Registros a procesar: {record_count:,}")

            if record_count == 0 and load_type == "INCREMENTAL":
                logger.info(f"  ⏭️  Sin registros nuevos. Saltando.")
                duration = time.time() - t0
                self._log_ingestion(
                    table_name=f"{BRONZE_PREFIX}{bronze_name}",
                    record_count=0, size_bytes=0, duration=duration,
                    status="SUCCESS", load_type="INCREMENTAL_SKIP",
                    watermark_value=watermark_value
                )
                return {
                    "table": f"{BRONZE_PREFIX}{bronze_name}",
                    "records": 0,
                    "duration_sec": round(duration, 2),
                    "status": "SUCCESS",
                    "load_type": "INCREMENTAL_SKIP"
                }

            # Agregar columnas de auditoría + partición
            df = self.add_audit_columns(df)

            # Guardar
            logger.info(f"  💾 Guardando en {bronze_table}...")
            if is_incremental:
                self._merge_incremental(df, bronze_table, primary_key, bronze_name)
            else:
                self._overwrite_full(df, bronze_table, bronze_name)

            duration = time.time() - t0
            size_bytes = get_delta_table_size(bronze_table)

            if is_incremental and incremental_col:
                new_wm = self._get_last_watermark(bronze_table, incremental_col)
                watermark_value = str(new_wm) if new_wm else watermark_value

            self._log_ingestion(
                table_name=f"{BRONZE_PREFIX}{bronze_name}",
                record_count=record_count, size_bytes=size_bytes,
                duration=duration, status="SUCCESS",
                load_type=load_type, watermark_value=watermark_value
            )

            logger.info(f"✅ Éxito | {record_count:,} registros | "
                        f"{size_bytes / (1024 ** 2):.2f} MB | {duration:.2f}s | {load_type}")

            return {
                "table": f"{BRONZE_PREFIX}{bronze_name}",
                "records": record_count,
                "size_mb": round(size_bytes / (1024 ** 2), 2),
                "duration_sec": round(duration, 2),
                "status": "SUCCESS",
                "load_type": load_type
            }

        except Exception as e:
            duration = time.time() - t0
            logger.error(f"❌ Error: {str(e)}", exc_info=True)
            self._log_ingestion(
                table_name=f"{BRONZE_PREFIX}{bronze_name}",
                record_count=0, size_bytes=0, duration=duration,
                status="FAILED", error_message=str(e),
                load_type=load_type, watermark_value=watermark_value
            )
            return {
                "table": f"{BRONZE_PREFIX}{bronze_name}",
                "status": "FAILED",
                "error": str(e)
            }

    def _merge_incremental(self, df_source, target_table, primary_key, bronze_name):
        """MERGE incremental con particionamiento"""
        merge_condition = f"target.{primary_key} = source.{primary_key}"
        table_exists = spark.catalog.tableExists(target_table)

        if table_exists:
            try:
                delta_table = DeltaTable.forName(spark, target_table)
                logger.info(f"  🔄 Tabla existente. Ejecutando MERGE...")

                (
                    delta_table.alias("target")
                    .merge(
                        df_source.alias("source"),
                        merge_condition
                    )
                    .whenMatchedUpdateAll()
                    .whenNotMatchedInsertAll()
                    .execute()
                )
                logger.info(f"  ✅ MERGE completado")

            except Exception as e:
                logger.warning(f"  ⚠️  Error en MERGE: {str(e)}")
                logger.info(f"  🔄 Fallback: APPEND...")
                df_source.write.format("delta") \
                    .mode("append") \
                    .saveAsTable(target_table)
                logger.info(f"  ✅ APPEND (fallback) completado")
        else:
            # Primera carga: crear tabla CON particiones
            logger.info(f"  🆕 Primera carga. Creando tabla particionada...")
            df_source.write.format("delta") \
                .mode("overwrite") \
                .partitionBy("_ingest_year", "_ingest_month", "_ingest_day") \
                .saveAsTable(target_table)
            logger.info(f"  ✅ Tabla creada: {target_table}")

    def _overwrite_full(self, df_source, target_table, bronze_name):
        """Overwrite completo con particionamiento"""
        logger.info(f"  🔄 Realizando OVERWRITE...")
        table_exists = spark.catalog.tableExists(target_table)

        if not table_exists:
            df_source.write.format("delta") \
                .mode("overwrite") \
                .partitionBy("_ingest_year", "_ingest_month", "_ingest_day") \
                .saveAsTable(target_table)
        else:
            df_source.write.format("delta") \
                .mode("overwrite") \
                .option("overwriteSchema", "true") \
                .saveAsTable(target_table)

        logger.info(f"  ✅ OVERWRITE completado → {target_table}")

    def _log_ingestion(self, table_name, record_count, size_bytes, duration,
                       status, error_message=None, load_type=None, watermark_value=None):
        """Registra log de auditoría"""
        try:
            log_row = [(
                datetime.now().isoformat(),
                self.batch_id,
                table_name,
                int(record_count),
                int(size_bytes),
                float(duration),
                status,
                error_message or "",
                load_type or "",
                watermark_value or ""
            )]
            log_df = spark.createDataFrame(log_row, AUDIT_SCHEMA)
            log_df.write.format("delta") \
                .mode("append") \
                .option("mergeSchema", "true") \
                .save(LAKEHOUSE_AUDIT)
        except Exception as e:
            logger.error(f"Error en log: {e}", exc_info=True)

    def ingest_all(self):
        """Ingestar todas las tablas"""
        results = []

        logger.info(f"\n{'=' * 80}")
        logger.info(f"🔄 INICIANDO INGESTA BRONZE")
        logger.info(f"{'=' * 80}")
        logger.info(f"\nBatch:     {self.batch_id}")
        logger.info(f"Origen:    {CATALOG}.{DATABASE}.{SCHEMA_ORIGEN}")
        logger.info(f"Destino:   Tables/ (schema: {BRONZE_SCHEMA}, prefijo: {BRONZE_PREFIX})")
        logger.info(f"Auditoría: Files/log_ingesta")
        logger.info(f"Columnas auditoría: _ingest_timestamp, _source_system, _batch_id")
        logger.info(f"Particiones: _ingest_year / _ingest_month / _ingest_day\n")

      #  for table_name, table_config in TABLES_CONFIG.items():
      #      result = self.ingest_table(table_name, table_config)
      #      results.append(result)
        for table_name, table_config in TABLES_CONFIG.items():

            def tarea():
                return self.ingest_table(table_name, table_config)

            ejecutar_con_manejo_errores(
                nombre_tarea=f"Ingesta_{table_name}",
                funcion=tarea
            )

        self._print_summary(results)
        return results

    def _print_summary(self, results):
        """Resumen final"""
        logger.info(f"\n{'=' * 80}")
        logger.info(f"📊 RESUMEN FINAL DE INGESTA BRONZE")
        logger.info(f"{'=' * 80}")

        success = len([r for r in results if r.get("status") == "SUCCESS"])
        failed = len([r for r in results if r.get("status") == "FAILED"])
        total_records = sum([r.get("records", 0) for r in results if r.get("status") == "SUCCESS"])
        total_size = sum([r.get("size_mb", 0) for r in results if r.get("status") == "SUCCESS"])
        total_duration = time.time() - self.execution_start.timestamp()

        logger.info(f"\n✅ Exitosas:        {success}/{len(results)}")
        logger.info(f"❌ Fallidas:        {failed}/{len(results)}")
        logger.info(f"📊 Total registros: {total_records:,}")
        logger.info(f"💾 Tamaño total:    {total_size:.2f} MB")
        logger.info(f"⏱️  Duración total:  {total_duration:.2f}s")

        logger.info(f"\n📋 Detalle:")
        logger.info(f"{'─' * 80}")
        logger.info(f"  {'Tabla':<35} {'Estado':<8} {'Registros':>10} {'MB':>8} {'Tipo':<15}")
        logger.info(f"{'─' * 80}")
        for r in results:
            if r.get("status") == "SUCCESS":
                logger.info(
                    f"  {r['table']:<35} ✅     "
                    f"{r.get('records', 0):>10,} "
                    f"{r.get('size_mb', 0):>8.2f} "
                    f"{r.get('load_type', ''):<15}"
                )
            else:
                logger.info(f"  {r['table']:<35} ❌ {r.get('error', '')[:35]}")
        logger.info(f"{'─' * 80}")
        logger.info(f"\n{'=' * 80}\n")


# ---------------------------
# Ejecución
# ---------------------------
#if __name__ == "__main__":
    if not globals().get("batch_id"):
        ts = spark.sql("SELECT date_format(current_timestamp(), 'yyyyMMdd_HHmmss') AS ts").collect()[0]["ts"]
        batch_id = f"batch_{ts}"
    else:
        batch_id = globals()["batch_id"]

    print(f"batch_id usado: {batch_id}")

    ingestion = BronzeIngestionFabric(batch_id=batch_id)
    results = ingestion.ingest_all()
    guardar_errores_pipeline()

StatementMeta(, 80202277-0337-4006-b2e8-3ba958c7f615, 10, Finished, Available, Finished, False)

batch_id usado: batch_20260407_104146
✅ Ingesta_tb_clientes_core
✅ Ingesta_tb_productos_cat
✅ Ingesta_tb_obligaciones
✅ Ingesta_tb_sucursales_red
✅ Ingesta_tb_comisiones_log

📋 pipeline_error_log: 6 OK, 0 errores


2026-04-07 11:01:42 - INFO - ✅ Tabla de auditoría existe
2026-04-07 11:01:42 - INFO - ================================================================================

2026-04-07 11:01:42 - INFO - 🚀 Iniciando ingesta Bronze | Batch: batch_20260407_104146
2026-04-07 11:01:42 - INFO - 
2026-04-07 11:01:42 - INFO - 🔄 INICIANDO INGESTA BRONZE
2026-04-07 11:01:42 - INFO - ================================================================================
2026-04-07 11:01:42 - INFO - 
Batch:     batch_20260407_104146
2026-04-07 11:01:42 - INFO - Origen:    Prueba Tecnica Dataknow.LH_BRONZE_PRUEBA.dbo
2026-04-07 11:01:42 - INFO - Destino:   Tables/ (schema: dbo, prefijo: BRZ_)
2026-04-07 11:01:42 - INFO - Auditoría: Files/log_ingesta
2026-04-07 11:01:42 - INFO - Columnas auditoría: _ingest_timestamp, _source_system, _batch_id
2026-04-07 11:01:42 - INFO - Particiones: _ingest_year / _ingest_month / _ingest_day

2026-04-07 11:01:42 - INFO - 
2026-04-07 11:01:42 - INFO - 📥 ORIGEN:  dbo.tb_clientes_